# Active Preference Learning via Polytope Volume Removal

Instead of heuristic sample filtering, we maintain the feasible set $\Omega_t$ as an **explicit polytope** $\{\omega : A\omega \leq b\}$.

Each pairwise response (left, right, indifferent, incomparable) adds **linear constraints** on $\omega$,
and we select queries to **maximize expected volume removal** (Sadigh et al. 2017).

### Constraint rules (from the frame model)

Given query gaps $\Delta_j$ and thresholds $\tau, \tau'$:

| Response | Constraints on $\omega$ |
|----------|------------------------|
| **Left** ($Y \succ Y'$) | $\sum \omega_j |\Delta_j| \geq \tau$ and $\sum \omega_j (\Delta_j - \tau' |\Delta_j|) \geq 0$ |
| **Right** ($Y \prec Y'$) | $\sum \omega_j |\Delta_j| \geq \tau$ and $\sum \omega_j (\Delta_j + \tau' |\Delta_j|) \leq 0$ |
| **Indifferent** ($Y \sim Y'$) | $\sum \omega_j |\Delta_j| \leq \tau - \eta$ |
| **Incomparable** ($Y \bowtie Y'$) | $\sum \omega_j |\Delta_j| \geq \tau$, $\sum \omega_j (\Delta_j - \tau' |\Delta_j|) < 0$, $\sum \omega_j (\Delta_j + \tau' |\Delta_j|) > 0$ |

In [42]:
import numpy as np
import pandas as pd
from scipy.optimize import linprog
from scipy.spatial.distance import pdist
from typing import List, Tuple, Optional, Set, Callable
from dataclasses import dataclass
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_style('whitegrid')

# ============================================================================
# Configuration
# ============================================================================

FEATURE_NAMES = ['elderlyDep', 'lifeYearsGained', 'obesity', 'weeklyWorkhours', 'yearsWaiting']

FEATURE_RANGES = {
    'elderlyDep': (0, 5),
    'lifeYearsGained': (0, 25),
    'obesity': (0, 5),
    'weeklyWorkhours': (0, 50),
    'yearsWaiting': (1, 10)
}

# Algorithm parameters (can be changed at test time)
TAU = 0.1           # Intensity threshold
TAU_PRIME = 0.1     # Resolvability threshold
LAMBDA_X = 1.0      # Query scaling factor

print(f'Features: {FEATURE_NAMES}')
print(f'tau={TAU}, tau_prime={TAU_PRIME}, lambda_x={LAMBDA_X}')


# ============================================================================
# Data structures
# ============================================================================

@dataclass
class Patient:
    """Represents a patient with feature values."""
    elderlyDep: float
    lifeYearsGained: float
    obesity: float
    weeklyWorkhours: float
    yearsWaiting: float

    def to_array(self) -> np.ndarray:
        """Convert to numpy array in standard feature order."""
        return np.array([
            self.elderlyDep,
            self.lifeYearsGained,
            self.obesity,
            self.weeklyWorkhours,
            self.yearsWaiting
        ], dtype=float)

    @classmethod
    def from_array(cls, arr: np.ndarray) -> 'Patient':
        """Create Patient from numpy array."""
        return cls(
            elderlyDep=float(arr[0]),
            lifeYearsGained=float(arr[1]),
            obesity=float(arr[2]),
            weeklyWorkhours=float(arr[3]),
            yearsWaiting=float(arr[4])
        )

    def __repr__(self):
        return f"Patient(elder={self.elderlyDep}, life={self.lifeYearsGained}, " \
               f"obesity={self.obesity}, work={self.weeklyWorkhours}, wait={self.yearsWaiting})"


@dataclass
class PairwiseQuery:
    """Represents a pairwise comparison query."""
    patient_left: Patient
    patient_right: Patient
    context: Optional[str] = None

    def __repr__(self):
        return f"Query:\n  LEFT:  {self.patient_left}\n  RIGHT: {self.patient_right}"


# ============================================================================
# Query generation
# ============================================================================

def generate_random_patient() -> Patient:
    """Generate a random patient with features in valid ranges."""
    return Patient(
        elderlyDep=np.random.randint(0, 5),
        lifeYearsGained=np.random.randint(0, 25),
        obesity=np.random.randint(0, 5),
        weeklyWorkhours=np.random.randint(0, 50),
        yearsWaiting=np.random.randint(1, 10)
    )


def generate_random_patient_normalized() -> Patient:
    """Generate a random patient with features normalized to [0, 1]."""
    return Patient(
        elderlyDep=np.random.uniform(0, 1),
        lifeYearsGained=np.random.uniform(0, 1),
        obesity=np.random.uniform(0, 1),
        weeklyWorkhours=np.random.uniform(0, 1),
        yearsWaiting=np.random.uniform(0, 1)
    )


def generate_candidate_queries_normalized(n_candidates: int = 50) -> List[PairwiseQuery]:
    """Generate candidate queries with features normalized to [0,1]."""
    candidates = []
    for _ in range(n_candidates):
        left = generate_random_patient_normalized()
        right = generate_random_patient_normalized()
        candidates.append(PairwiseQuery(left, right))
    return candidates


print("Data structures defined: Patient, PairwiseQuery")
print("Query generators: generate_random_patient, generate_random_patient_normalized, generate_candidate_queries_normalized")


Features: ['elderlyDep', 'lifeYearsGained', 'obesity', 'weeklyWorkhours', 'yearsWaiting']
tau=0.1, tau_prime=0.1, lambda_x=1.0
Data structures defined: Patient, PairwiseQuery
Query generators: generate_random_patient, generate_random_patient_normalized, generate_candidate_queries_normalized


In [ ]:
# ============================================================================
# Core Frame Model Computations
# ============================================================================

def compute_frame_gaps(
    query: PairwiseQuery,
    lambda_x: float = LAMBDA_X,
    V: np.ndarray = None,
    tau: float = TAU
) -> Tuple[np.ndarray, Set[int]]:
    """
    Compute frame-level gaps and identify active (decisive) frames.

    Parameters
    ----------
    query : PairwiseQuery
        The comparison query
    lambda_x : float
        Scaling factor for feature differences
    V : np.ndarray, optional
        Sign matrix (diagonal) to handle negative weights. 
        V[i,i] = -1 if oracle weight i is negative, else +1.
    tau : float
        Threshold for determining active frames

    Returns
    -------
    gaps : np.ndarray, shape (n_frames,)
        Gap for each frame: lambda_x * V @ (left_j - right_j)
    active_frames : Set[int]
        Frames where |gap_j| > tau (frames that "speak" to this query)
    """
    left_features = query.patient_left.to_array()
    right_features = query.patient_right.to_array()
    feature_diff = left_features - right_features
    if V is not None:
        gaps = lambda_x * (V @ feature_diff)
    else:
        gaps = lambda_x * feature_diff
    active_frames = set(np.where(np.abs(gaps) > 0)[0].tolist())
    return gaps, active_frames


def compute_aggregate_scores(
    gaps: np.ndarray,
    weights: np.ndarray,
    active_frames: Set[int]
) -> Tuple[float, float]:
    """
    Compute aggregate preference score delta(omega) and intensity r(omega).

    Parameters
    ----------
    gaps : np.ndarray
        Per-frame gaps from compute_frame_gaps
    weights : np.ndarray
        Weight vector omega (on simplex)
    active_frames : Set[int]
        Which frames are active for this query

    Returns
    -------
    delta_omega : float
        Weighted sum of gaps: sum_j omega_j * gap_j
        Positive = left is better, Negative = right is better
    r_omega : float
        Weighted sum of absolute gaps: sum_j omega_j * |gap_j|
        Measures "intensity" - how strongly the frames speak
    """
    if len(active_frames) == 0:
        return 0.0, 0.0

    active_list = sorted(list(active_frames))
    active_gaps = gaps[active_list]
    active_weights = weights[active_list]

    delta_omega = np.dot(active_weights, active_gaps)
    r_omega = np.dot(active_weights, np.abs(active_gaps))

    return delta_omega, r_omega


def predict_response(
    query: PairwiseQuery,
    weights: np.ndarray,
    tau: float = TAU,
    lambda_x: float = LAMBDA_X,
    tau_prime: float = TAU_PRIME,
    V: np.ndarray = None
) -> str:
    """
    Predict response for a query given a weight vector (DETERMINISTIC).

    Parameters
    ----------
    query : PairwiseQuery
        The comparison query
    weights : np.ndarray
        Weight vector omega (on simplex, all non-negative)
    tau : float
        Intensity threshold
    lambda_x : float
        Scaling factor
    tau_prime : float
        Resolvability threshold
    V : np.ndarray, optional
        Sign matrix (diagonal) to handle negative oracle weights.

    Decision rule:
    - r < tau           -> 'indifferent'  (not enough intensity)
    - |delta| < tau'*r  -> 'incomparable' (frames disagree)
    - delta >= tau'*r   -> 'left'         (left is better)
    - delta <= -tau'*r  -> 'right'        (right is better)

    Returns one of: 'left', 'right', 'indifferent', 'incomparable'
    """
    gaps, active_frames = compute_frame_gaps(query, lambda_x=lambda_x, V=V, tau=tau)
    delta_omega, r_omega = compute_aggregate_scores(gaps, weights, active_frames)

    if r_omega < tau:
        return 'indifferent'
    if r_omega >= tau and np.abs(delta_omega) < tau_prime * r_omega:
        return 'incomparable'
    elif r_omega >= tau and delta_omega >= tau_prime * r_omega:
        return 'left'
    elif r_omega >= tau and delta_omega <= -tau_prime * r_omega:
        return 'right'
    else:
        return 'indifferent'


def make_sign_matrix(beta: np.ndarray) -> np.ndarray:
    """
    Create diagonal sign matrix V from weight vector beta.
    
    V[i,i] = -1 if beta[i] < 0, else +1
    
    This allows the polytope algorithm (which requires non-negative weights)
    to handle negative oracle weights by absorbing signs into V.
    
    To recover signed weights: beta = V @ omega (where omega >= 0)
    """
    return np.diag(np.sign(beta))


print("Core functions defined:")
print("  - compute_frame_gaps(query, lambda_x, V, tau) -> (gaps, active_frames)")
print("  - compute_aggregate_scores(gaps, weights, active_frames) -> (delta_omega, r_omega)")
print("  - predict_response(query, weights, tau, lambda_x, tau_prime, V) -> response")
print("  - make_sign_matrix(beta) -> V (diagonal sign matrix)")

In [44]:
# ============================================================================
# Noise Injection for Latent Margins
# ============================================================================
#
# Key insight: Bradley-Terry and Thurstone-Mosteller models can be viewed as
# adding noise to the latent preference score (delta_omega) before deciding.
#
# When tau = tau' = 0:
#   - The decision simplifies to: left if delta > 0, right if delta < 0
#   - Adding logistic noise: P(left) = sigmoid(delta/scale) = Bradley-Terry
#   - Adding normal noise: P(left) = Phi(delta/scale) = Thurstone-Mosteller
#
# With tau, tau' > 0, we still get the full 4-response frame model,
# but with stochastic responses near the decision boundaries.
# ============================================================================

def no_noise(delta_omega: float, r_omega: float) -> Tuple[float, float]:
    """
    No noise injection - returns latent margins unchanged.
    
    This gives the deterministic frame model.
    """
    return delta_omega, r_omega


def logistic_noise(scale: float = 1.0) -> Callable:
    """
    Factory for logistic noise injection (Bradley-Terry equivalent).

    When tau=tau'=0, this gives:
        P(left) = sigmoid(delta_omega / scale)

    which is exactly the Bradley-Terry model.

    Parameters
    ----------
    scale : float
        Scale parameter of the logistic distribution.
        Larger scale = more noise = more randomness in responses.
        
    Returns
    -------
    noise_fn : callable(delta_omega, r_omega) -> (delta_tilde, r_tilde)
    """
    def noise_fn(delta_omega: float, r_omega: float) -> Tuple[float, float]:
        # Logistic(0, scale) has CDF = sigmoid(x/scale)
        # So P(delta + eta > 0) = P(eta > -delta) = sigmoid(delta/scale)
        eta = np.random.logistic(loc=0, scale=scale)
        return delta_omega + eta, r_omega
    
    return noise_fn


def normal_noise(scale: float = 1.0) -> Callable:
    """
    Factory for normal/Gaussian noise injection (Thurstone-Mosteller equivalent).

    When tau=tau'=0, this gives:
        P(left) = Phi(delta_omega / scale)

    which is exactly the Thurstone-Mosteller (Case V) model.

    Parameters
    ----------
    scale : float
        Standard deviation of the normal distribution.
        Larger scale = more noise = more randomness in responses.

    Returns
    -------
    noise_fn : callable(delta_omega, r_omega) -> (delta_tilde, r_tilde)
    """
    def noise_fn(delta_omega: float, r_omega: float) -> Tuple[float, float]:
        eta = np.random.normal(loc=0, scale=scale)
        return delta_omega + eta, r_omega
    
    return noise_fn


def predict_response_noisy(
    query: PairwiseQuery,
    weights: np.ndarray,
    noise_fn: Callable = no_noise,
    tau: float = TAU,
    lambda_x: float = LAMBDA_X,
    tau_prime: float = TAU_PRIME,
) -> str:
    """
    Predict response with noise injection into latent margins.

    This function:
    1. Computes the latent margins (delta_omega, r_omega)
    2. Applies noise_fn to get (delta_tilde, r_tilde)
    3. Applies the frame decision rule to the noisy margins

    Parameters
    ----------
    query : PairwiseQuery
        The comparison query
    weights : np.ndarray
        Weight vector omega on the simplex
    noise_fn : callable
        Noise injection function: (delta, r) -> (delta_tilde, r_tilde)
        Use no_noise for deterministic, logistic_noise(scale) for BT,
        or normal_noise(scale) for Thurstone-Mosteller.
    tau : float
        Intensity threshold for indifference
    lambda_x : float
        Feature scaling factor
    tau_prime : float
        Resolvability threshold for incomparability

    Returns
    -------
    response : str
        One of 'left', 'right', 'indifferent', 'incomparable'

    Notes
    -----
    When tau=tau'=0:
    - With logistic_noise(scale=1), this equals Bradley-Terry
    - With normal_noise(scale=1), this equals Thurstone-Mosteller
    """
    # Step 1: Compute gaps and aggregate scores
    gaps, active_frames = compute_frame_gaps(query, lambda_x, tau)
    delta_omega, r_omega = compute_aggregate_scores(gaps, weights, active_frames)

    # Step 2: Apply noise injection
    delta_tilde, r_tilde = noise_fn(delta_omega, r_omega)

    # Step 3: Apply decision rule to noisy margins
    if r_tilde < tau:
        return 'indifferent'
    if r_tilde >= tau and np.abs(delta_tilde) < tau_prime * r_tilde:
        return 'incomparable'
    elif r_tilde >= tau and delta_tilde >= tau_prime * r_tilde:
        return 'left'
    elif r_tilde >= tau and delta_tilde <= -tau_prime * r_tilde:
        return 'right'
    else:
        return 'indifferent'


print("Noise injection functions defined:")
print("  - no_noise: deterministic (no noise)")
print("  - logistic_noise(scale): BT-equivalent when tau=tau'=0")
print("  - normal_noise(scale): TM-equivalent when tau=tau'=0")
print("  - predict_response_noisy(query, weights, noise_fn, ...): noisy response")


Noise injection functions defined:
  - no_noise: deterministic (no noise)
  - logistic_noise(scale): BT-equivalent when tau=tau'=0
  - normal_noise(scale): TM-equivalent when tau=tau'=0
  - predict_response_noisy(query, weights, noise_fn, ...): noisy response


In [45]:
# ============================================================================
# Test: Verify BT Equivalence
# ============================================================================
# When tau=tau'=0 with logistic noise, predict_response_noisy should give
# the same probability distribution as Bradley-Terry.

from collections import Counter
from scipy.special import expit  # sigmoid function

np.random.seed(42)

# Create a simple test query
left = Patient(elderlyDep=0.8, lifeYearsGained=0.6, obesity=0.3, weeklyWorkhours=0.5, yearsWaiting=0.4)
right = Patient(elderlyDep=0.2, lifeYearsGained=0.4, obesity=0.7, weeklyWorkhours=0.5, yearsWaiting=0.6)
query = PairwiseQuery(left, right)

# Weight vector
weights = np.array([0.1, 0.5, 0.1, 0.1, 0.2])

# Compute delta_omega (what BT uses)
delta_x = left.to_array() - right.to_array()
delta_omega = np.dot(weights, delta_x)

print("Test Setup:")
print(f"  delta_x (left - right): {delta_x}")
print(f"  weights: {weights}")
print(f"  delta_omega = w . delta_x = {delta_omega:.4f}")
print()

# BT theoretical probability
bt_prob_left = expit(delta_omega)  # sigmoid(delta_omega)
print(f"Bradley-Terry P(left) = sigmoid({delta_omega:.4f}) = {bt_prob_left:.4f}")
print()

# Empirical test: run many trials with tau=tau'=0 and logistic noise
n_trials = 10000
responses = []
for _ in range(n_trials):
    r = predict_response_noisy(
        query, weights, 
        noise_fn=logistic_noise(scale=1.0),
        tau=0.0, tau_prime=0.0, lambda_x=1.0
    )
    responses.append(r)

counts = Counter(responses)
empirical_prob_left = counts['left'] / n_trials

print(f"Empirical test ({n_trials} trials, tau=tau'=0, logistic noise):")
print(f"  Response counts: {dict(counts)}")
print(f"  Empirical P(left) = {empirical_prob_left:.4f}")
print()
print(f"Match: BT={bt_prob_left:.4f} vs Empirical={empirical_prob_left:.4f}")
print(f"  Difference: {abs(bt_prob_left - empirical_prob_left):.4f}")
if abs(bt_prob_left - empirical_prob_left) < 0.02:
    print("  ✓ Close match! Frame model with logistic noise ≈ Bradley-Terry")
else:
    print("  ✗ Mismatch - something is wrong")


Test Setup:
  delta_x (left - right): [ 0.6  0.2 -0.4  0.  -0.2]
  weights: [0.1 0.5 0.1 0.1 0.2]
  delta_omega = w . delta_x = 0.0800

Bradley-Terry P(left) = sigmoid(0.0800) = 0.5200

Empirical test (10000 trials, tau=tau'=0, logistic noise):
  Response counts: {'right': 4864, 'left': 5136}
  Empirical P(left) = 0.5136

Match: BT=0.5200 vs Empirical=0.5136
  Difference: 0.0064
  ✓ Close match! Frame model with logistic noise ≈ Bradley-Terry


In [46]:
# ============================================================================
# Likelihood Computation for Noisy Frame Model (Fixed)
# ============================================================================

from scipy.special import expit as sigmoid
from scipy.stats import logistic as logistic_dist, norm as normal_dist


def compute_response_log_likelihood(
    query: PairwiseQuery,
    response: str,
    omega: np.ndarray,
    noise_type: str = 'none',
    scale: float = 1.0,
    tau: float = TAU,
    tau_prime: float = TAU_PRIME,
    lambda_x: float = LAMBDA_X,
) -> float:
    """
    Compute log P(response | query, omega) for the noisy frame model.
    """
    # Step 1: Compute latent margins
    gaps, active_frames = compute_frame_gaps(query, lambda_x, tau)
    delta, r = compute_aggregate_scores(gaps, omega, active_frames)
    
    # Step 2: Deterministic case
    if noise_type == 'none':
        predicted = predict_response(query, omega, tau, lambda_x, tau_prime)
        return 0.0 if response == predicted else -np.inf
    
    # Step 3: r < tau => always indifferent (deterministic, no noise on r)
    if r < tau:
        return 0.0 if response == 'indifferent' else -np.inf
    
    # Step 4: r >= tau, compute probabilities
    # Thresholds for noise variable eta:
    #   left:  eta >= tau'*r - delta
    #   right: eta <= -tau'*r - delta
    threshold_left = tau_prime * r - delta
    threshold_right = -tau_prime * r - delta
    
    if noise_type == 'logistic':
        # P(left) = P(eta >= threshold_left) = 1 - sigmoid(threshold_left/scale)
        p_left = 1.0 - sigmoid(threshold_left / scale)
        # P(right) = P(eta <= threshold_right) = sigmoid(threshold_right/scale)
        p_right = sigmoid(threshold_right / scale)
        # P(incomparable) = P(threshold_right < eta < threshold_left)
        p_incomp = sigmoid(threshold_left / scale) - sigmoid(threshold_right / scale)
        
    elif noise_type == 'normal':
        p_left = 1.0 - normal_dist.cdf(threshold_left / scale)
        p_right = normal_dist.cdf(threshold_right / scale)
        p_incomp = normal_dist.cdf(threshold_left / scale) - normal_dist.cdf(threshold_right / scale)
    else:
        raise ValueError(f"Unknown noise_type: {noise_type}")
    
    # P(indifferent) = 0 when r >= tau (by construction)
    # Don't clip this - it's genuinely 0
    
    # Return log probability
    if response == 'left':
        return np.log(max(p_left, 1e-15))
    elif response == 'right':
        return np.log(max(p_right, 1e-15))
    elif response == 'incomparable':
        # p_incomp can be negative due to numerical issues when tau'=0
        # In that case, it should be 0
        return np.log(max(p_incomp, 1e-15))
    elif response == 'indifferent':
        # When r >= tau, indifferent is impossible
        return -np.inf
    else:
        raise ValueError(f"Unknown response: {response}")


def compute_transcript_log_likelihood(
    transcript: List[Tuple[PairwiseQuery, str]],
    omega: np.ndarray,
    noise_type: str = 'logistic',
    scale: float = 1.0,
    tau: float = TAU,
    tau_prime: float = TAU_PRIME,
    lambda_x: float = LAMBDA_X,
) -> float:
    """Compute total log-likelihood of a transcript."""
    ll = 0.0
    for query, response in transcript:
        ll += compute_response_log_likelihood(
            query, response, omega, noise_type, scale, tau, tau_prime, lambda_x
        )
    return ll


In [47]:
# ============================================================================
# Bayesian Inference via Hit-and-Run MCMC with Metropolis-Hastings
# ============================================================================

def hit_and_run_simplex_step(x: np.ndarray) -> np.ndarray:
    """
    One hit-and-run step on the simplex {w : sum(w)=1, w>=0}.
    
    1. Pick random direction in tangent space (sum=0)
    2. Find chord bounds where x + t*d stays in simplex
    3. Sample uniformly along chord
    """
    dim = len(x)
    
    # Random direction projected onto sum=0 hyperplane
    d = np.random.randn(dim)
    d = d - d.mean()  # Project to sum=0
    norm = np.linalg.norm(d)
    if norm < 1e-12:
        return x.copy()
    d = d / norm
    
    # Find t bounds: x + t*d >= 0 for all components
    t_min = -np.inf
    t_max = np.inf
    
    for j in range(dim):
        if d[j] > 1e-12:
            # x[j] + t*d[j] >= 0  =>  t >= -x[j]/d[j]
            t_min = max(t_min, -x[j] / d[j])
        elif d[j] < -1e-12:
            # x[j] + t*d[j] >= 0  =>  t <= -x[j]/d[j]
            t_max = min(t_max, -x[j] / d[j])
    
    # Check for valid interval
    if t_min >= t_max - 1e-12:
        return x.copy()
    
    # Sample uniformly from [t_min, t_max]
    t = np.random.uniform(t_min, t_max)
    new_x = x + t * d
    
    # Assert feasibility within tolerance, else reject by returning x
    if new_x.min() < -1e-10:
        return x.copy()
    s = new_x.sum()
    if abs(s - 1.0) > 1e-10:
        return x.copy()
    # numerical cleanup
    new_x = np.maximum(new_x, 0.0)
    new_x = new_x / new_x.sum()
    
    return new_x


def sample_posterior_hit_and_run(
    transcript: List[Tuple[PairwiseQuery, str]],
    noise_type: str = 'logistic',
    scale: float = 1.0,
    tau: float = TAU,
    tau_prime: float = TAU_PRIME,
    lambda_x: float = LAMBDA_X,
    n_samples: int = 2000,
    burn_in: int = 1000,
    thin: int = 1,
    verbose: bool = True,
) -> Tuple[np.ndarray, float]:
    """
    Sample from posterior P(omega | transcript) using hit-and-run + MH.
    
    Returns (samples, acceptance_rate)
    """
    dim = len(FEATURE_NAMES)
    
    # Initialize at simplex center
    omega = np.ones(dim) / dim
    ll_current = compute_transcript_log_likelihood(
        transcript, omega, noise_type, scale, tau, tau_prime, lambda_x
    )
    
    samples = []
    n_accepted = 0
    total_steps = burn_in + n_samples * thin
    
    for step in range(total_steps):
        # Hit-and-run proposal
        proposal = hit_and_run_simplex_step(omega)
        
        # Compute proposal likelihood
        ll_proposal = compute_transcript_log_likelihood(
            transcript, proposal, noise_type, scale, tau, tau_prime, lambda_x
        )
        
        # MH acceptance (symmetric proposal, uniform prior)
        log_alpha = ll_proposal - ll_current
        
        if np.log(np.random.rand()) < log_alpha:
            omega = proposal
            ll_current = ll_proposal
            if step >= burn_in:
                n_accepted += 1
        
        # Collect sample
        if step >= burn_in and (step - burn_in) % thin == 0:
            samples.append(omega.copy())
    
    acceptance_rate = n_accepted / (n_samples * thin)
    if verbose:
        print(f"MCMC: {len(samples)} samples, acceptance = {acceptance_rate:.1%}, final LL = {ll_current:.2f}")
    
    return np.array(samples), acceptance_rate


def bayesian_frame_estimate(
    transcript: List[Tuple[PairwiseQuery, str]],
    noise_type: str = 'logistic',
    scale: float = 1.0,
    tau: float = TAU,
    tau_prime: float = TAU_PRIME,
    lambda_x: float = LAMBDA_X,
    n_samples: int = 2000,
    burn_in: int = 1000,
    verbose: bool = True,
) -> dict:
    """
    Bayesian estimation of omega. Reports posterior mean (like BT's MLE).
    """
    samples, acc_rate = sample_posterior_hit_and_run(
        transcript, noise_type, scale, tau, tau_prime, lambda_x,
        n_samples, burn_in, verbose=verbose
    )
    
    return {
        'posterior_mean': samples.mean(axis=0),
        'posterior_std': samples.std(axis=0),
        'samples': samples,
        'acceptance_rate': acc_rate,
    }


print("Bayesian inference defined (hit-and-run + MH)")


Bayesian inference defined (hit-and-run + MH)


In [48]:
# ============================================================================
# Active Learning with BALD Query Selection
# ============================================================================
#
# BALD (Bayesian Active Learning by Disagreement):
# Select query that maximizes I(y; ω | q, D) = H[E_ω[p(y|q,ω)]] - E_ω[H[p(y|q,ω)]]
#
# This is the mutual information between the response y and weights ω,
# which measures how much we expect to learn about ω from the response.
# ============================================================================

def compute_response_probs_fast(query, omega, noise_type, scale, tau, tau_prime, lambda_x):
    gaps, active_frames = compute_frame_gaps(query, lambda_x, tau)
    delta, r = compute_aggregate_scores(gaps, omega, active_frames)
    probs = np.zeros(4)
    if r < tau:
        probs[2] = 1.0  # indifferent
        return probs
    thr_L = tau_prime * r - delta
    thr_R = -tau_prime * r - delta
    if noise_type == 'logistic':
        sL = sigmoid(thr_L / scale)
        sR = sigmoid(thr_R / scale)
        probs[0] = 1.0 - sL         # left
        probs[1] = sR               # right
        probs[3] = sL - sR          # incomparable
    else:  # normal
        pL = 1.0 - normal_dist.cdf(thr_L / scale)
        pR = normal_dist.cdf(thr_R / scale)
        probs[0] = pL
        probs[1] = pR
        probs[3] = (normal_dist.cdf(thr_L / scale) - normal_dist.cdf(thr_R / scale))
    probs = np.clip(probs, 0.0, 1.0)
    probs /= probs.sum() + 1e-15
    return probs



def entropy(probs: np.ndarray) -> float:
    """Compute entropy H[p] = -sum(p * log(p))"""
    probs = np.clip(probs, 1e-15, 1.0)
    return -np.sum(probs * np.log(probs))


def bald_score(
    query: PairwiseQuery,
    posterior_samples: np.ndarray,
    noise_type: str,
    scale: float,
    tau: float,
    tau_prime: float,
    lambda_x: float,
    max_samples: int = 100,
) -> float:
    """
    Compute BALD score for a query.
    
    BALD = H[E_ω[p(y|q,ω)]] - E_ω[H[p(y|q,ω)]]
         = (entropy of average prediction) - (average entropy of predictions)
    
    Higher BALD = more informative query.
    """
    n_samples = min(len(posterior_samples), max_samples)
    
    # Compute p(y|q,ω) for each posterior sample
    all_probs = []
    for omega in posterior_samples[:n_samples]:
        probs = compute_response_probs_fast(
            query, omega, noise_type, scale, tau, tau_prime, lambda_x
        )
        all_probs.append(probs)
    
    all_probs = np.array(all_probs)  # (n_samples, 4)
    
    # E_ω[p(y|q,ω)] = average prediction
    avg_probs = all_probs.mean(axis=0)
    
    # H[E_ω[p(y|q,ω)]] = entropy of average
    H_avg = entropy(avg_probs)
    
    # E_ω[H[p(y|q,ω)]] = average of entropies
    avg_H = np.mean([entropy(p) for p in all_probs])
    
    # BALD = mutual information
    return H_avg - avg_H


def active_learning_bayesian(
    oracle_weights: np.ndarray,
    noise_type: str = 'logistic',
    scale: float = 1.0,
    tau: float = TAU,
    tau_prime: float = TAU_PRIME,
    lambda_x: float = LAMBDA_X,
    max_iterations: int = 50,
    n_candidates: int = 100,
    n_posterior_samples: int = 1000,
    target_diameter: float = 0.1,
    verbose: bool = True,
) -> dict:
    """
    Active learning with BALD query selection.
    """
    dim = len(FEATURE_NAMES)
    transcript = []
    history = []
    
    # Oracle noise function
    if noise_type == 'none':
        oracle_noise_fn = no_noise
    elif noise_type == 'logistic':
        oracle_noise_fn = logistic_noise(scale)
    elif noise_type == 'normal':
        oracle_noise_fn = normal_noise(scale)
    else:
        raise ValueError(f"Unknown noise_type: {noise_type}")
    
    if verbose:
        print("=" * 60)
        print("Bayesian Active Learning with BALD")
        print("=" * 60)
        print(f"Oracle: {oracle_weights}")
        print(f"Noise: {noise_type}, scale={scale}, tau={tau}, tau'={tau_prime}")
        print()
    
    for iteration in range(max_iterations):
        # Get posterior samples
        if len(transcript) == 0:
            samples = np.random.dirichlet(np.ones(dim), size=n_posterior_samples)
        else:
            samples, acc_rate = sample_posterior_hit_and_run(
                transcript, noise_type, scale, tau, tau_prime, lambda_x,
                n_samples=n_posterior_samples, burn_in=500, verbose=False
            )
        
        # Compute metrics
        if len(samples) > 100:
            idx = np.random.choice(len(samples), 100, replace=False)
            diam = pdist(samples[idx], metric='cityblock').max()
        else:
            diam = pdist(samples, metric='cityblock').max()
        
        posterior_mean = samples.mean(axis=0)
        cos_sim = np.dot(posterior_mean, oracle_weights) / (
            np.linalg.norm(posterior_mean) * np.linalg.norm(oracle_weights) + 1e-10
        )
        l1_err = np.abs(posterior_mean - oracle_weights).sum()
        
        if verbose:
            print(f"Iter {iteration+1:2d}: diam={diam:.3f}, cos={cos_sim:.3f}, L1={l1_err:.3f}")
        
        history.append({
            'iteration': iteration + 1,
            'diameter': diam,
            'cosine_similarity': cos_sim,
            'l1_error': l1_err,
            'posterior_mean': posterior_mean.copy(),
        })
        
        if diam <= target_diameter:
            if verbose:
                print(f"Converged!")
            break
        
        # Generate candidates and select using BALD
        candidates = generate_candidate_queries_normalized(n_candidates)
        
        best_query = None
        best_bald = -np.inf
        
        for query in candidates:
            score = bald_score(
                query, samples, noise_type, scale, tau, tau_prime, lambda_x,
                max_samples=100
            )
            if score > best_bald:
                best_bald = score
                best_query = query
        
        if best_query is None:
            best_query = candidates[0]
        
        # Oracle response
        response = predict_response_noisy(
            best_query, oracle_weights, oracle_noise_fn, tau, lambda_x, tau_prime
        )
        transcript.append((best_query, response))
        
        if verbose and iteration < 5:
            print(f"       BALD={best_bald:.3f}, response={response}")
    
    # Final result
    if len(transcript) > 0:
        final_samples, _ = sample_posterior_hit_and_run(
            transcript, noise_type, scale, tau, tau_prime, lambda_x,
            n_samples=n_posterior_samples, burn_in=500, verbose=False
        )
    else:
        final_samples = samples
    
    final_mean = final_samples.mean(axis=0)
    
    if verbose:
        print()
        print("=" * 60)
        print(f"Final: {len(transcript)} queries")
        print(f"Oracle:    {oracle_weights}")
        print(f"Estimate:  {final_mean}")
        print(f"L1 error:  {np.abs(final_mean - oracle_weights).sum():.4f}")
    
    return {
        'transcript': transcript,
        'posterior_mean': final_mean,
        'posterior_samples': final_samples,
        'history': history,
    }


print("Active learning with BALD defined:")
print("  - compute_response_probs(query, omega, ...): P(y|q,ω) for all y")
print("  - bald_score(query, samples, ...): mutual information score")
print("  - active_learning_bayesian(...): uses BALD for query selection")


Active learning with BALD defined:
  - compute_response_probs(query, omega, ...): P(y|q,ω) for all y
  - bald_score(query, samples, ...): mutual information score
  - active_learning_bayesian(...): uses BALD for query selection


In [49]:
# ============================================================================
# TEST: More queries, less noise
# ============================================================================

np.random.seed(42)
oracle_weights = np.array([0.1, 0.5, 0.1, 0.1, 0.2])

print("Testing weight recovery with different settings:")
print("=" * 70)
print()

for n_queries in [20, 50, 100]:
    for noise_scale in [1.0, 0.5, 0.25]:
        # Generate transcript
        queries = [generate_candidate_queries_normalized(1)[0] for _ in range(n_queries)]
        noise_fn = logistic_noise(scale=noise_scale)
        transcript = [(q, predict_response_noisy(q, oracle_weights, noise_fn, 
                                                  tau=0.0, tau_prime=0.0)) 
                      for q in queries]
        
        # Check response distribution
        from collections import Counter
        counts = Counter(r for q, r in transcript)
        
        # Compute likelihoods
        ll_oracle = compute_transcript_log_likelihood(
            transcript, oracle_weights, 'logistic', noise_scale, 0.0, 0.0, 1.0)
        ll_uniform = compute_transcript_log_likelihood(
            transcript, np.ones(5)/5, 'logistic', noise_scale, 0.0, 0.0, 1.0)
        
        # Run MCMC
        samples, acc_rate = sample_posterior_hit_and_run(
            transcript, 'logistic', noise_scale, 0.0, 0.0, 1.0,
            n_samples=1000, burn_in=500, verbose=False
        )
        posterior_mean = samples.mean(axis=0)
        l1_error = np.abs(posterior_mean - oracle_weights).sum()
        
        oracle_better = "✓" if ll_oracle > ll_uniform else "✗"
        
        print(f"n={n_queries:3d}, scale={noise_scale:.2f}: "
              f"LL_oracle-LL_uniform={ll_oracle-ll_uniform:+.2f} {oracle_better}, "
              f"L1_error={l1_error:.3f}, acc={acc_rate:.1%}")

print()
print("Key insight: More queries + less noise = better recovery")
print("With scale=1.0 and few queries, uniform weights can explain noise better!")


Testing weight recovery with different settings:

n= 20, scale=1.00: LL_oracle-LL_uniform=-0.42 ✗, L1_error=0.682, acc=95.9%
n= 20, scale=0.50: LL_oracle-LL_uniform=-0.67 ✗, L1_error=0.691, acc=91.2%
n= 20, scale=0.25: LL_oracle-LL_uniform=+1.63 ✓, L1_error=0.333, acc=80.6%
n= 50, scale=1.00: LL_oracle-LL_uniform=+0.26 ✓, L1_error=0.583, acc=94.8%
n= 50, scale=0.50: LL_oracle-LL_uniform=+2.58 ✓, L1_error=0.190, acc=80.1%
n= 50, scale=0.25: LL_oracle-LL_uniform=+0.91 ✓, L1_error=0.566, acc=75.7%
n=100, scale=1.00: LL_oracle-LL_uniform=+0.10 ✓, L1_error=0.611, acc=94.6%
n=100, scale=0.50: LL_oracle-LL_uniform=-0.51 ✗, L1_error=0.557, acc=88.8%
n=100, scale=0.25: LL_oracle-LL_uniform=+3.51 ✓, L1_error=0.118, acc=70.0%

Key insight: More queries + less noise = better recovery
With scale=1.0 and few queries, uniform weights can explain noise better!


In [50]:
# ============================================================================
# TEST: Scale matching between BT and our model
# ============================================================================
# 
# Key insight: BT's implicit scale depends on the MAGNITUDE of weights.
# On the simplex (sum=1), our deltas are small, so we need smaller scale.
#
# Rule of thumb: If typical |delta| ≈ 0.3 on simplex, we need scale ≈ 0.3
# to get similar noise-to-signal ratio as BT with scale=1 and |delta| ≈ 1.
# ============================================================================

from scipy.optimize import minimize
import numpy as np

def bt_mle(
    transcript,
    dim: int,
    *,
    lambda_x: float = 1.0,
    scale: float = 1.0,
    l2_theta: float = 0.0,
    seed: int | None = None,
    n_restarts: int = 10,
):
    """
    Bradley–Terry / logistic-regression MLE (or MAP if l2_theta>0) with a simplex-constrained
    weight vector w via a softmax parameterization.

    This is the apples-to-apples BT baseline for your tau=tau'=0 frame model when your
    likelihood is:
        P(left | q, w) = sigmoid( (lambda_x/scale) * (x_L - x_R)^T w )

    Notes
    -----
    - Uses ONLY 'left'/'right' rows (BT is binary).
    - Constrains w to the simplex by w = softmax(theta).
    - Matches your frame model's effective logit scaling via (lambda_x/scale).
    - l2_theta adds a Gaussian prior on theta (MAP), set to 0 for pure MLE.
    """
    rng = np.random.default_rng(seed)

    # Build design matrix on differences and binary labels
    X, y = [], []
    for query, response in transcript:
        if response not in ("left", "right"):
            continue
        delta = query.patient_left.to_array() - query.patient_right.to_array()
        X.append(delta)
        y.append(1.0 if response == "left" else 0.0)

    if len(X) == 0:
        return np.ones(dim) / dim

    X = np.asarray(X, dtype=float)
    y = np.asarray(y, dtype=float)

    # Safe sigmoid
    def sigmoid(z):
        # numerically stable sigmoid
        z = np.asarray(z)
        out = np.empty_like(z, dtype=float)
        pos = z >= 0
        out[pos] = 1.0 / (1.0 + np.exp(-z[pos]))
        ez = np.exp(z[~pos])
        out[~pos] = ez / (1.0 + ez)
        return out

    def softmax(theta):
        t = theta - np.max(theta)
        e = np.exp(t)
        return e / np.sum(e)

    # Negative log-likelihood in theta-space
    # logits = (lambda_x/scale) * X @ w, w=softmax(theta)
    temp = lambda_x / scale

    def neg_log_post(theta):
        w = softmax(theta)
        logits = temp * (X @ w)
        p = sigmoid(logits)

        # Bernoulli log-likelihood, stable
        eps = 1e-12
        ll = np.sum(y * np.log(p + eps) + (1.0 - y) * np.log(1.0 - p + eps))

        # Optional L2 regularization on theta (MAP, not pure MLE)
        reg = l2_theta * np.sum(theta * theta) if l2_theta > 0 else 0.0
        return -ll + reg

    best = None
    for _ in range(max(1, n_restarts)):
        theta0 = rng.normal(0.0, 0.1, size=dim)
        res = minimize(neg_log_post, theta0, method="L-BFGS-B")
        if best is None or res.fun < best.fun:
            best = res

    w_hat = softmax(best.x)
    return w_hat



np.random.seed(42)
oracle_weights = np.array([0.1, 0.5, 0.1, 0.1, 0.2])
n_queries = 100

# Generate queries and compute typical delta magnitude
queries = [generate_candidate_queries_normalized(1)[0] for _ in range(n_queries)]
deltas = [np.dot(oracle_weights, q.patient_left.to_array() - q.patient_right.to_array()) 
          for q in queries]
typical_delta = np.mean(np.abs(deltas))
print(f"Typical |delta| on simplex: {typical_delta:.3f}")
print(f"Suggested scale for similar SNR: {typical_delta:.3f}")
print()

# Compare different scales
print("=" * 70)
print(f"{'Scale':<10} {'BT L1 err':<12} {'Bayes L1 err':<14} {'Match?':<10}")
print("=" * 70)

for scale in [1.0, 0.5, 0.3, 0.2, 0.1]:
    noise_fn = logistic_noise(scale=scale)
    transcript = [(q, predict_response_noisy(q, oracle_weights, noise_fn, 
                                              tau=0.0, tau_prime=0.0)) 
                  for q in queries]
    
    # BT MLE (doesn't use scale - it's implicit in the data)
    bt_est = bt_mle(
    transcript,
    dim=len(FEATURE_NAMES),
    lambda_x=LAMBDA_X,
    scale=scale,        # SAME scale you use in the frame likelihood
    l2_theta=0.0,       # set >0 only if you want MAP
    seed=0,
    n_restarts=10,
)
    bt_l1 = np.abs(bt_est - oracle_weights).sum()
    
    # Our Bayes estimate (uses scale in likelihood)
    samples, _ = sample_posterior_hit_and_run(
        transcript, 'logistic', scale, 0.0, 0.0, 1.0,
        n_samples=1000, burn_in=500, verbose=False
    )
    bayes_est = samples.mean(axis=0)
    bayes_l1 = np.abs(bayes_est - oracle_weights).sum()
    
    match = "✓" if abs(bt_l1 - bayes_l1) < 0.1 else ""
    print(f"{scale:<10.2f} {bt_l1:<12.4f} {bayes_l1:<14.4f} {match:<10}")

print()
print("When scale matches the typical delta magnitude, BT and Bayes should agree!")


Typical |delta| on simplex: 0.204
Suggested scale for similar SNR: 0.204

Scale      BT L1 err    Bayes L1 err   Match?    
1.00       1.4000       0.6870                   
0.50       0.6269       0.3937                   
0.30       0.2116       0.1164         ✓         
0.20       0.1834       0.1773         ✓         
0.10       0.3492       0.2823         ✓         

When scale matches the typical delta magnitude, BT and Bayes should agree!


In [51]:
# ============================================================================
# COMPARE: β_eff for BT vs Bayes
# ============================================================================
# β_eff = (λ_x / scale) * ω
# This is what actually matters for predictions
# ============================================================================

from scipy.optimize import minimize


np.random.seed(42)
oracle_weights = np.array([0.1, 0.5, 0.1, 0.1, 0.2])
scale = 0.3
LAMBDA_X = 1.0
n_queries = 100

# Generate data
queries = [generate_candidate_queries_normalized(1)[0] for _ in range(n_queries)]
noise_fn = logistic_noise(scale=scale)
transcript = [(q, predict_response_noisy(q, oracle_weights, noise_fn, tau=0.0, tau_prime=0.0)) 
              for q in queries]

print(f"Generated {n_queries} queries with scale={scale}, λ_x={LAMBDA_X}")
print(f"Oracle ω: {oracle_weights}")
print()

# BT MLE (on simplex)
w_bt = bt_mle(
            transcript,
            dim=len(FEATURE_NAMES),
            lambda_x=LAMBDA_X,
            scale=scale,        # SAME scale you use in the frame likelihood
            l2_theta=0.0,       # set >0 only if you want MAP
            seed=0,
            n_restarts=10,
        )
print(f"BT MLE ω:     {w_bt}")

# Bayes posterior mean
samples, acc = sample_posterior_hit_and_run(
    transcript, 'logistic', scale, 0.0, 0.0, LAMBDA_X,
    n_samples=2000, burn_in=1000, verbose=False
)
omega_posterior_mean = samples.mean(axis=0)
print(f"Bayes mean ω: {omega_posterior_mean}")
print()

# Compute β_eff
beta_eff_oracle = (LAMBDA_X / scale) * oracle_weights
beta_eff_bt = (LAMBDA_X / scale) * w_bt
beta_eff_bayes = (LAMBDA_X / scale) * omega_posterior_mean

print("=" * 70)
print("β_eff = (λ_x / scale) * ω")
print("=" * 70)
print(f"Oracle β_eff: {beta_eff_oracle}")
print(f"BT β_eff:     {beta_eff_bt}")
print(f"Bayes β_eff:  {beta_eff_bayes}")
print()

# Compare
print("=" * 70)
print("COMPARISON")
print("=" * 70)

# L1 distance
l1_bt = np.abs(beta_eff_bt - beta_eff_oracle).sum()
l1_bayes = np.abs(beta_eff_bayes - beta_eff_oracle).sum()
print(f"L1 distance to oracle β_eff:")
print(f"  BT:    {l1_bt:.4f}")
print(f"  Bayes: {l1_bayes:.4f}")
print()

# Cosine similarity
def cosine(a, b):
    return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b) + 1e-10)

cos_bt = cosine(beta_eff_bt, beta_eff_oracle)
cos_bayes = cosine(beta_eff_bayes, beta_eff_oracle)
print(f"Cosine similarity to oracle β_eff:")
print(f"  BT:    {cos_bt:.4f}")
print(f"  Bayes: {cos_bayes:.4f}")
print()

# Compare predictions on test queries
print("=" * 70)
print("PREDICTION COMPARISON (the real test!)")
print("=" * 70)
test_queries = [generate_candidate_queries_normalized(1)[0] for _ in range(100)]
test_X = np.array([q.patient_left.to_array() - q.patient_right.to_array() for q in test_queries])

# P(left) = sigmoid(β_eff · delta_x)
p_oracle = sigmoid(test_X @ beta_eff_oracle)
p_bt = sigmoid(test_X @ beta_eff_bt)
p_bayes = sigmoid(test_X @ beta_eff_bayes)

print(f"Mean |P_pred - P_oracle| on 100 test queries:")
print(f"  BT:    {np.mean(np.abs(p_bt - p_oracle)):.4f}")
print(f"  Bayes: {np.mean(np.abs(p_bayes - p_oracle)):.4f}")
print()

print(f"Correlation with oracle P(left):")
print(f"  BT:    {np.corrcoef(p_bt, p_oracle)[0,1]:.4f}")
print(f"  Bayes: {np.corrcoef(p_bayes, p_oracle)[0,1]:.4f}")
print()

print(f"BT vs Bayes agreement:")
print(f"  Mean |P_bt - P_bayes|: {np.mean(np.abs(p_bt - p_bayes)):.4f}")
print(f"  Correlation:           {np.corrcoef(p_bt, p_bayes)[0,1]:.4f}")


Generated 100 queries with scale=0.3, λ_x=1.0
Oracle ω: [0.1 0.5 0.1 0.1 0.2]

BT MLE ω:     [0.04848486 0.26434651 0.27218586 0.1200827  0.29490007]
Bayes mean ω: [0.12027768 0.24315319 0.2589582  0.15477958 0.22283136]

β_eff = (λ_x / scale) * ω
Oracle β_eff: [0.33333333 1.66666667 0.33333333 0.33333333 0.66666667]
BT β_eff:     [0.16161618 0.88115504 0.9072862  0.40027565 0.98300025]
Bayes β_eff:  [0.40092558 0.81051063 0.863194   0.51593192 0.74277119]

COMPARISON
L1 distance to oracle β_eff:
  BT:    1.9145
  Bayes: 1.7123

Cosine similarity to oracle β_eff:
  BT:    0.8355
  Bayes: 0.8384

PREDICTION COMPARISON (the real test!)
Mean |P_pred - P_oracle| on 100 test queries:
  BT:    0.0755
  Bayes: 0.0793

Correlation with oracle P(left):
  BT:    0.8562
  Bayes: 0.8431

BT vs Bayes agreement:
  Mean |P_bt - P_bayes|: 0.0274
  Correlation:           0.9758


In [52]:
# ============================================================================
# PROPER COMPARISON: BT vs Bayes with matched scale
# ============================================================================
# 
# Key insight: The scale parameter must be CONSISTENT between:
# 1. Data generation (noise_fn)
# 2. Likelihood computation (for Bayes)
# 3. BT's implicit scale (absorbed in weight magnitude)
#
# On the simplex, typical |delta| ≈ 0.3, so we should use scale ≈ 0.3
# ============================================================================



np.random.seed(42)
oracle_weights = np.array([0.1, 0.5, 0.1, 0.1, 0.2])

# Compute typical delta to set appropriate scale
test_queries = [generate_candidate_queries_normalized(1)[0] for _ in range(100)]
deltas = [abs(np.dot(oracle_weights, q.patient_left.to_array() - q.patient_right.to_array())) 
          for q in test_queries]
typical_delta = np.mean(deltas)
print(f"Typical |delta| on simplex: {typical_delta:.3f}")
print(f"Recommended scale: ~{typical_delta:.2f} (so SNR ≈ 1)")
print()

# Test with different scales
print("=" * 75)
print(f"{'Scale':<8} {'N':<6} {'BT L1':<10} {'Bayes L1':<10} {'BT cos':<10} {'Bayes cos':<10}")
print("=" * 75)

queries_50 = [generate_candidate_queries_normalized(1)[0] for _ in range(50)]


for scale in [0.5, 0.3, 0.2]:
    queries = queries_50
    noise_fn = logistic_noise(scale=scale)
    transcript = [(q, predict_response_noisy(q, oracle_weights, noise_fn, 
                                                tau=0.0, tau_prime=0.0)) 
                    for q in queries]
    
    # BT MLE
    bt_est = bt_mle(
        transcript,
        dim=len(FEATURE_NAMES),
        lambda_x=LAMBDA_X,
        scale=scale,        # SAME scale you use in the frame likelihood
        l2_theta=0.0,       # set >0 only if you want MAP
        seed=0,
        n_restarts=10,
    )
    bt_l1 = np.abs(bt_est - oracle_weights).sum()
    bt_cos = np.dot(bt_est, oracle_weights) / (np.linalg.norm(bt_est) * np.linalg.norm(oracle_weights))
    
    # Bayes (use SAME scale for likelihood)
    samples, _ = sample_posterior_hit_and_run(
        transcript, 'logistic', scale, 0.0, 0.0, 1.0,
        n_samples=2000, burn_in=1000, verbose=False
    )
    bayes_est = samples.mean(axis=0)
    bayes_l1 = np.abs(bayes_est - oracle_weights).sum()
    bayes_cos = np.dot(bayes_est, oracle_weights) / (np.linalg.norm(bayes_est) * np.linalg.norm(oracle_weights))
    
    print(f"{scale:<8.2f} {n_queries:<6} {bt_l1:<10.4f} {bayes_l1:<10.4f} {bt_cos:<10.4f} {bayes_cos:<10.4f}")

print()
print("With appropriate scale (≈ typical delta), both methods should work better!")
print("Bayes has advantage: prior regularization prevents overfitting to noise.")


Typical |delta| on simplex: 0.204
Recommended scale: ~0.20 (so SNR ≈ 1)

Scale    N      BT L1      Bayes L1   BT cos     Bayes cos 
0.50     100    0.7368     0.4021     0.8096     0.9116    
0.30     100    0.4945     0.3927     0.8995     0.9194    
0.20     100    0.3873     0.3673     0.9176     0.9213    

With appropriate scale (≈ typical delta), both methods should work better!
Bayes has advantage: prior regularization prevents overfitting to noise.


## Polytope class

H-representation: $\Omega = \{\omega \in \mathbb{R}^d : A\omega \leq b\}$

With hit-and-run MCMC sampling and Chebyshev center computation.

In [53]:
class ConstraintPolytope:
    """Convex polytope in H-representation: {w : Aw <= b}.

    For the simplex, we store the equality sum(w)=1 separately and
    handle it via two inequality constraints (sum <= 1, -sum <= -1).
    The Chebyshev center LP uses a tolerance approach to find a
    strictly interior point of the inequality system.
    """

    def __init__(self, dim: int, geometry: str = 'simplex'):
        self.dim = dim
        self.geometry = geometry
        self._A_rows = []
        self._b_vals = []
        self._center_cache = None
        self._use_sphere_constraint = False

        if geometry == 'simplex':
            # w_j >= 0  =>  -w_j <= 0
            for j in range(dim):
                row = np.zeros(dim)
                row[j] = -1.0
                self._A_rows.append(row)
                self._b_vals.append(0.0)
            # sum w_j <= 1
            self._A_rows.append(np.ones(dim))
            self._b_vals.append(1.0)
            # sum w_j >= 1  =>  -sum w_j <= -1
            self._A_rows.append(-np.ones(dim))
            self._b_vals.append(-1.0)

        elif geometry == 'sphere':
            for j in range(dim):
                row_pos = np.zeros(dim)
                row_pos[j] = 1.0
                self._A_rows.append(row_pos)
                self._b_vals.append(1.0)
                row_neg = np.zeros(dim)
                row_neg[j] = -1.0
                self._A_rows.append(row_neg)
                self._b_vals.append(1.0)
            self._use_sphere_constraint = True
        else:
            raise ValueError(f'Unknown geometry: {geometry}')

    @property
    def A(self) -> np.ndarray:
        return np.array(self._A_rows)

    @property
    def b(self) -> np.ndarray:
        return np.array(self._b_vals)

    @property
    def n_constraints(self) -> int:
        return len(self._A_rows)

    def add_constraint(self, a: np.ndarray, b_val: float):
        """Add constraint a^T w <= b_val."""
        self._A_rows.append(a.copy())
        self._b_vals.append(b_val)
        self._center_cache = None

    def is_feasible(self, w: np.ndarray, tol: float = 1e-8) -> bool:
        """Check if w satisfies all constraints."""
        violations = self.A @ w - self.b
        if np.any(violations > tol):
            return False
        if self._use_sphere_constraint and np.linalg.norm(w) > 1.0 + tol:
            return False
        return True

    def chebyshev_center(self) -> Optional[np.ndarray]:
        """Find a strictly interior point of the polytope.

        For polytopes with equality constraints (like the simplex where
        sum=1), the classical Chebyshev center has radius 0. Instead we
        find the point that maximizes the minimum slack across all
        INEQUALITY constraints (excluding exact equalities).

        We detect near-equalities (pairs a^T x <= b and -a^T x <= -b)
        and handle them as equality constraints in the LP.
        """
        if self._center_cache is not None:
            return self._center_cache.copy()

        A = self.A
        b_vec = self.b
        m, d = A.shape

        # Detect equality pairs: rows i,j where A[i] ≈ -A[j] and b[i] ≈ -b[j]
        eq_rows = set()
        ineq_rows = list(range(m))
        for i in range(m):
            for j in range(i + 1, m):
                if (np.allclose(A[i], -A[j], atol=1e-10) and
                        abs(b_vec[i] + b_vec[j]) < 1e-10):
                    eq_rows.add(i)
                    eq_rows.add(j)

        ineq_rows = [i for i in range(m) if i not in eq_rows]
        eq_row_list = sorted(eq_rows)

        if len(ineq_rows) == 0:
            # Only equalities — just solve for feasibility
            # Use one row from each equality pair
            seen = set()
            A_eq_rows = []
            b_eq_vals = []
            for i in eq_row_list:
                key = tuple(np.round(A[i], 10))
                neg_key = tuple(np.round(-A[i], 10))
                if key not in seen and neg_key not in seen:
                    seen.add(key)
                    A_eq_rows.append(A[i])
                    b_eq_vals.append(b_vec[i])
            A_eq = np.array(A_eq_rows)
            b_eq = np.array(b_eq_vals)
            # Least-norm solution
            x, _, _, _ = np.linalg.lstsq(A_eq, b_eq, rcond=None)
            if self.is_feasible(x):
                self._center_cache = x
                return x.copy()
            return None

        # Build LP: max r s.t. a_i^T x + r ||a_i|| <= b_i (for ineq rows)
        #           a_j^T x = b_j (for equality rows)
        A_ineq = A[ineq_rows]
        b_ineq = b_vec[ineq_rows]

        norms = np.linalg.norm(A_ineq, axis=1, keepdims=True)
        # Avoid division issues for zero-norm rows
        norms = np.maximum(norms, 1e-15)

        # Variables: [x_1, ..., x_d, r]
        c_obj = np.zeros(d + 1)
        c_obj[-1] = -1.0  # maximize r

        A_lp = np.hstack([A_ineq, norms])
        b_lp = b_ineq

        # Equality constraints from detected pairs (keep one per pair)
        A_eq_lp = None
        b_eq_lp = None
        if len(eq_row_list) > 0:
            seen = set()
            A_eq_rows = []
            b_eq_vals = []
            for i in eq_row_list:
                key = tuple(np.round(A[i], 10))
                neg_key = tuple(np.round(-A[i], 10))
                if key not in seen and neg_key not in seen:
                    seen.add(key)
                    A_eq_rows.append(np.append(A[i], 0.0))  # r doesn't appear
                    b_eq_vals.append(b_vec[i])
            if A_eq_rows:
                A_eq_lp = np.array(A_eq_rows)
                b_eq_lp = np.array(b_eq_vals)

        bounds = [(None, None)] * d + [(0, None)]

        result = linprog(c_obj, A_ub=A_lp, b_ub=b_lp,
                         A_eq=A_eq_lp, b_eq=b_eq_lp,
                         bounds=bounds, method='highs')

        if result.success:
            center = result.x[:d]
            if self.is_feasible(center):
                self._center_cache = center
                return center.copy()

        # Fallback: try the centroid (1/d, ..., 1/d) for simplex
        if self.geometry == 'simplex':
            centroid = np.ones(d) / d
            if self.is_feasible(centroid):
                self._center_cache = centroid
                return centroid.copy()

        return None

    def _hit_and_run_step(self, x: np.ndarray) -> np.ndarray:
        """One hit-and-run step: random direction, find chord, sample uniformly."""
        A = self.A
        b_vec = self.b

        direction = np.random.randn(self.dim)
        direction /= np.linalg.norm(direction)

        # For simplex: project direction onto the sum=1 hyperplane
        if self.geometry == 'simplex':
            direction -= direction.mean()
            norm = np.linalg.norm(direction)
            if norm < 1e-15:
                return x
            direction /= norm

        # Find t range: A(x + t*d) <= b  =>  t*(Ad) <= b - Ax
        Ad = A @ direction
        residuals = b_vec - A @ x

        t_min = -np.inf
        t_max = np.inf

        for i in range(len(Ad)):
            if Ad[i] > 1e-12:
                t_max = min(t_max, residuals[i] / Ad[i])
            elif Ad[i] < -1e-12:
                t_min = max(t_min, residuals[i] / Ad[i])

        if t_min >= t_max - 1e-15:
            return x

        # For sphere constraint, also clip t range
        if self._use_sphere_constraint:
            a_coef = np.dot(direction, direction)
            b_coef = 2 * np.dot(x, direction)
            c_coef = np.dot(x, x) - 1.0
            disc = b_coef**2 - 4 * a_coef * c_coef
            if disc > 0:
                sqrt_disc = np.sqrt(disc)
                t_lo = (-b_coef - sqrt_disc) / (2 * a_coef)
                t_hi = (-b_coef + sqrt_disc) / (2 * a_coef)
                t_min = max(t_min, t_lo)
                t_max = min(t_max, t_hi)
            else:
                return x

        if t_min >= t_max - 1e-15:
            return x

        t = np.random.uniform(t_min, t_max)
        return x + t * direction

    def sample(self, n_samples: int, burn_in: int = 500, thin: int = 10) -> np.ndarray:
        """Sample from the polytope using hit-and-run MCMC."""
        center = self.chebyshev_center()
        if center is None:
            raise ValueError('Polytope appears empty (no Chebyshev center found)')

        x = center.copy()
        samples = []

        total_steps = burn_in + n_samples * thin
        for step in range(total_steps):
            x = self._hit_and_run_step(x)
            if step >= burn_in and (step - burn_in) % thin == 0:
                samples.append(x.copy())

        return np.array(samples)


# Quick test
poly = ConstraintPolytope(5, geometry='simplex')
center = poly.chebyshev_center()
print(f'Simplex Chebyshev center: {center}')
print(f'Sum = {center.sum():.4f}, all >= 0: {np.all(center >= -1e-10)}')
print(f'Initial constraints: {poly.n_constraints}')

samples = poly.sample(500, burn_in=200, thin=5)
print(f'\nSampled {len(samples)} points from simplex')
print(f'  Sum range: [{samples.sum(axis=1).min():.6f}, {samples.sum(axis=1).max():.6f}]')
print(f'  Min weight: {samples.min():.4f}')
print(f'  Mean: {samples.mean(axis=0)}')

Simplex Chebyshev center: [0.2 0.2 0.2 0.2 0.2]
Sum = 1.0000, all >= 0: True
Initial constraints: 7

Sampled 500 points from simplex
  Sum range: [1.000000, 1.000000]
  Min weight: 0.0001
  Mean: [0.18147144 0.20543988 0.20002102 0.20070997 0.21235769]


## Constraint generation from responses

Convert each (query, response) pair into linear constraints $a^\top \omega \leq b$.

In [54]:
def response_to_constraints(
    query: PairwiseQuery,
    response: str,
    tau: float = TAU,
    tau_prime: float = TAU_PRIME,
    lambda_x: float = LAMBDA_X,
) -> List[Tuple[np.ndarray, float]]:
    """
    Convert a (query, response) into linear constraints on omega.

    Returns list of (a, b) where each represents: a^T omega <= b.

    Only active frames (|Δ_j| >= tau) participate.

    For strict inequalities (incomparable, indifferent), we use the
    non-strict form (<=, >=) since the boundary has measure zero and
    doesn't affect volume. This avoids the need for an eta parameter
    that could accidentally exclude the true weights.

    Left:          r >= tau  AND  Delta >= tau' * r
    Right:         r >= tau  AND  Delta <= -tau' * r
    Indifferent:   r <= tau  (non-strict relaxation of r < tau)
    Incomparable:  r >= tau  AND  Delta <= tau' * r  AND  Delta >= -tau' * r
    """
    gaps, active_frames = compute_frame_gaps(query, lambda_x, tau)
    abs_gaps = np.abs(gaps)

    # Zero out inactive frames
    mask = np.zeros(len(gaps))
    for j in active_frames:
        mask[j] = 1.0
    gaps_active = gaps * mask
    abs_gaps_active = abs_gaps * mask

    constraints = []

    if response == 'left':
        # r >= tau  =>  -|Δ|^T ω <= -tau
        constraints.append((-abs_gaps_active, -tau))
        # Delta >= tau' * r  =>  -(Δ - τ'|Δ|)^T ω <= 0
        constraints.append((-(gaps_active - tau_prime * abs_gaps_active), 0.0))

    elif response == 'right':
        # r >= tau
        constraints.append((-abs_gaps_active, -tau))
        # Delta <= -tau' * r  =>  (Δ + τ'|Δ|)^T ω <= 0
        constraints.append((gaps_active + tau_prime * abs_gaps_active, 0.0))

    elif response == 'indifferent':
        # r <= tau  (non-strict: boundary has measure zero)
        constraints.append((abs_gaps_active, tau))

    elif response == 'incomparable':
        # r >= tau
        constraints.append((-abs_gaps_active, -tau))
        # Delta <= tau' * r  (non-strict)  =>  (Δ - τ'|Δ|)^T ω <= 0
        constraints.append((gaps_active - tau_prime * abs_gaps_active, 0.0))
        # Delta >= -tau' * r  (non-strict)  =>  -(Δ + τ'|Δ|)^T ω <= 0
        constraints.append((-(gaps_active + tau_prime * abs_gaps_active), 0.0))

    else:
        raise ValueError(f'Unknown response: {response}')

    return constraints


def classify_samples_active(
    samples: np.ndarray,
    query: PairwiseQuery,
    tau: float = TAU,
    tau_prime: float = TAU_PRIME,
    lambda_x: float = LAMBDA_X,
) -> np.ndarray:
    """
    Classify each sample using active-frame logic (matching predict_response).
    """
    gaps, active_frames = compute_frame_gaps(query, lambda_x, tau)

    if len(active_frames) == 0:
        return np.full(len(samples), 'indifferent', dtype=object)

    active_list = sorted(list(active_frames))
    active_gaps = gaps[active_list]
    abs_active_gaps = np.abs(active_gaps)
    active_weights = samples[:, active_list]

    r_vals = active_weights @ abs_active_gaps
    delta_vals = active_weights @ active_gaps

    N = len(samples)
    responses = np.empty(N, dtype=object)

    intense = r_vals >= tau
    responses[~intense] = 'indifferent'

    strongly_left = intense & (delta_vals >= tau_prime * r_vals)
    strongly_right = intense & (delta_vals <= -tau_prime * r_vals)
    incomparable = intense & ~strongly_left & ~strongly_right

    responses[strongly_left] = 'left'
    responses[strongly_right] = 'right'
    responses[incomparable] = 'incomparable'

    return responses


# Verify: oracle weights must satisfy constraints they generate
oracle_w_test = np.array([0.1, 0.5, 0.1, 0.1, 0.2])

test_left = Patient(0.6, 0.8, 0.2, 0.5, 0.4)
test_right = Patient(0.2, 0.2, 0.6, 0.3, 0.8)
test_query = PairwiseQuery(test_left, test_right)

resp = predict_response(test_query, oracle_w_test)
gaps_t, active_t = compute_frame_gaps(test_query, LAMBDA_X, TAU)
print(f'Test gaps: {gaps_t}')
print(f'Active frames (|gap| >= {TAU}): {sorted(active_t)}')
print(f'Oracle response: {resp}')

cs = response_to_constraints(test_query, resp)
print(f'\nConstraints ({len(cs)}):')
all_satisfied = True
for a, bv in cs:
    val = np.dot(a, oracle_w_test)
    ok = val <= bv + 1e-10
    all_satisfied &= ok
    print(f'  a^T w = {val:.6f} <= {bv:.6f}  {"OK" if ok else "VIOLATED!"}')
print(f'\nOracle satisfies all constraints: {all_satisfied}')

Test gaps: [ 0.4  0.6 -0.4  0.2 -0.4]
Active frames (|gap| >= 0.1): [0, 1, 2, 3, 4]
Oracle response: left

Constraints (2):
  a^T w = -0.480000 <= -0.100000  OK
  a^T w = -0.192000 <= 0.000000  OK

Oracle satisfies all constraints: True


In [55]:
def generate_axis_aligned_queries(n):
    queries = []
    for _ in range(n):
        i = np.random.randint(0, 5)
        delta = np.zeros(5)
        delta[i] = np.random.choice([-1.0, 1.0])
        xL = np.clip(0.5 + 0.5 * delta, 0, 1)
        xR = np.clip(0.5 - 0.5 * delta, 0, 1)
        queries.append(PairwiseQuery(Patient(*xL), Patient(*xR)))
    return queries


In [ ]:
# ============================================================================
# STEP 1: Load data and extract the 60 pairwise comparisons for one person
# ============================================================================

import pandas as pd
import numpy as np
from scipy.special import expit as sigmoid

np.random.seed(42)

# Load data
df = pd.read_csv('kidney_pairwise_data.csv')

# Feature info
feature_names = ['elderlyDep', 'lifeYearsGained', 'obesity', 'weeklyWorkhours', 'yearsWaiting']
left_cols = [f'l_{f}' for f in feature_names]
right_cols = [f'r_{f}' for f in feature_names]

# Feature ranges for normalization
feature_ranges = {
    'elderlyDep': (0, 3),
    'lifeYearsGained': (5, 25),
    'obesity': (0, 5),
    'weeklyWorkhours': (0, 50),
    'yearsWaiting': (1, 7)
}

def normalize_features(row, cols):
    """Normalize features to [0, 1]."""
    normalized = []
    for col in cols:
        feat = col[2:]  # remove 'l_' or 'r_' prefix
        lo, hi = feature_ranges[feat]
        val = (float(row[col]) - lo) / (hi - lo)
        normalized.append(np.clip(val, 0, 1))
    return np.array(normalized)

# Pick one person-session
session_counts = df.groupby(['id', 'sessionid']).size().reset_index(name='count')
good_sessions = session_counts[session_counts['count'] >= 50]
sample = good_sessions.iloc[0]
sample_id, sample_session = sample['id'], sample['sessionid']

person_df = df[(df['id'] == sample_id) & (df['sessionid'] == sample_session)].copy()
print(f"Person id={sample_id}, session={sample_session}")
print(f"Number of pairwise comparisons: {len(person_df)}")

# Build normalized feature differences for each comparison
comparisons = []
for idx, row in person_df.iterrows():
    left = normalize_features(row, left_cols)
    right = normalize_features(row, right_cols)
    delta = left - right  # feature difference
    comparisons.append({
        'comparison_id': len(comparisons),
        'delta': delta,
        'left_features': left,
        'right_features': right
    })

print(f"\nFirst 3 comparisons (normalized feature differences):")
for i in range(3):
    print(f"  Comparison {i}: Δ = {comparisons[i]['delta'].round(3)}")

In [ ]:
# ============================================================================
# STEP 2: Set oracle weight vector and create sign matrix V
# ============================================================================

# Oracle weight vector β* (ground truth)
# These represent how much the decision-maker values each feature
# Positive = prefer higher values, Negative = prefer lower values
beta_oracle = np.array([
    -0.5,   # elderlyDep: prefer fewer elderly dependents
    +1.0,   # lifeYearsGained: strongly prefer more life years
    -0.3,   # obesity: slightly prefer lower obesity
    -0.2,   # weeklyWorkhours: slightly prefer lower work hours  
    +0.8    # yearsWaiting: prefer longer wait time (more "deserving")
])

print("Oracle weight vector β*:")
print("-" * 50)
for name, beta in zip(feature_names, beta_oracle):
    direction = "prefer HIGHER" if beta > 0 else "prefer LOWER"
    print(f"  {name:20s}: {beta:+.2f}  ({direction})")

# Create sign matrix V
# V is diagonal with V[i,i] = sign(β*[i])
# This allows polytope algorithm to work with non-negative ω where β* = V @ ω
V = np.diag(np.sign(beta_oracle))
omega_oracle = np.abs(beta_oracle)  # The non-negative version for polytope

print("\n" + "-" * 50)
print("Sign matrix V (diagonal):")
print(f"  diag(V) = {np.diag(V)}")
print(f"\nNon-negative ω = |β*| = {omega_oracle}")
print(f"Verify: V @ ω = β*: {V @ omega_oracle}")

print("\n" + "=" * 60)
print("GENERATING BTL RESPONSES")
print("=" * 60)
print("""
Bradley-Terry-Luce Model:
  P(choose left | β*, Δ) = sigmoid(β*^T Δ)
                         = sigmoid((V @ ω)^T Δ)
                         = sigmoid(ω^T (V @ Δ))
  
where Δ = x_left - x_right (normalized feature differences)

The sign matrix V flips the sign of Δ for features with negative β*.
""")

# Generate responses for each comparison
transcript = []
for comp in comparisons:
    delta = comp['delta']
    
    # Compute preference score: β*^T Δ = (V @ ω)^T Δ = ω^T (V @ Δ)
    score = np.dot(beta_oracle, delta)
    
    # BTL probability of choosing left
    prob_left = sigmoid(score)
    
    # Sample response
    chose_left = np.random.random() < prob_left
    response = 'LEFT' if chose_left else 'RIGHT'
    
    transcript.append({
        'comparison_id': comp['comparison_id'],
        'delta': delta,
        'V_delta': V @ delta,  # Store transformed delta for polytope
        'score': score,
        'prob_left': prob_left,
        'response': response
    })

# Display transcript
print(f"Generated {len(transcript)} responses:\n")
print(f"{'ID':>3} | {'Score':>7} | {'P(left)':>7} | {'Response':>8}")
print("-" * 35)
for t in transcript[:15]:  # Show first 15
    print(f"{t['comparison_id']:3d} | {t['score']:+7.3f} | {t['prob_left']:7.3f} | {t['response']:>8}")
print("...")

# Summary statistics
left_count = sum(1 for t in transcript if t['response'] == 'LEFT')
right_count = len(transcript) - left_count
print(f"\nSummary: LEFT={left_count}, RIGHT={right_count}")

In [ ]:
# ============================================================================
# STEP 3: Polytope Algorithm on BTL Transcript (with noise handling)
# ============================================================================
#
# Problem: BTL responses are noisy/probabilistic, so we may get "inconsistent"
# responses that would make the polytope empty if we add hard constraints.
#
# Solution: Check feasibility before adding each constraint. If a constraint
# would make the polytope empty, skip it (the response was "noise").
#
# For BTL:
#   - LEFT chosen  => we believe (V @ Δ)^T ω > 0  => constraint: -(V @ Δ)^T ω <= 0
#   - RIGHT chosen => we believe (V @ Δ)^T ω < 0  => constraint:  (V @ Δ)^T ω <= 0
#
# Note: We work with ω (non-negative) and use V @ Δ to absorb the signs.
# ============================================================================

def btl_response_to_constraint(V_delta: np.ndarray, response: str) -> Tuple[np.ndarray, float]:
    """
    Convert BTL response to a single linear constraint on ω.
    
    Parameters
    ----------
    V_delta : np.ndarray
        V @ (x_left - x_right), the sign-adjusted feature difference
    response : str
        'LEFT' or 'RIGHT'
    
    Returns
    -------
    (a, b) : constraint a^T ω <= b
    """
    if response == 'LEFT':
        # We believe V_delta^T ω > 0, i.e., -(V_delta)^T ω < 0
        # Constraint: -(V_delta)^T ω <= 0
        return (-V_delta, 0.0)
    elif response == 'RIGHT':
        # We believe V_delta^T ω < 0, i.e., (V_delta)^T ω < 0
        # Constraint: (V_delta)^T ω <= 0
        return (V_delta, 0.0)
    else:
        raise ValueError(f"Unknown response: {response}")


def check_feasibility_with_constraint(polytope, a, b, tol=1e-8):
    """
    Check if polytope remains feasible after adding constraint a^T ω <= b.
    Uses LP to find a feasible point in the augmented system.
    """
    from scipy.optimize import linprog
    
    # Current constraints
    A_current = polytope.A
    b_current = polytope.b
    
    # Add new constraint
    A_new = np.vstack([A_current, a.reshape(1, -1)])
    b_new = np.append(b_current, b)
    
    # Try to find any feasible point (minimize 0)
    dim = polytope.dim
    result = linprog(
        c=np.zeros(dim),
        A_ub=A_new,
        b_ub=b_new,
        bounds=[(0, None)] * dim,  # ω >= 0
        method='highs'
    )
    
    return result.success


# Initialize polytope on simplex
dim = len(feature_names)
polytope = ConstraintPolytope(dim, geometry='simplex')

print("POLYTOPE ALGORITHM ON BTL TRANSCRIPT")
print("=" * 60)
print(f"Initial polytope: {dim}-simplex with {polytope.n_constraints} constraints")
print(f"Oracle ω = |β*| = {omega_oracle}")
print(f"Sign matrix diag(V) = {np.diag(V)}")
print()

# Process each response in the transcript
n_added = 0
n_skipped = 0
skipped_ids = []

print("Processing responses...")
print("-" * 60)

for t in transcript:
    comp_id = t['comparison_id']
    V_delta = t['V_delta']
    response = t['response']
    
    # Get constraint for this response
    a, b_val = btl_response_to_constraint(V_delta, response)
    
    # Check if adding this constraint keeps polytope feasible
    if check_feasibility_with_constraint(polytope, a, b_val):
        polytope.add_constraint(a, b_val)
        n_added += 1
        status = "ADDED"
    else:
        n_skipped += 1
        skipped_ids.append(comp_id)
        status = "SKIPPED (would cause infeasibility)"
    
    # Print first 10 and any skipped
    if comp_id < 10 or status.startswith("SKIPPED"):
        print(f"  Comparison {comp_id:2d}: {response:5s} -> {status}")

print("-" * 60)
print(f"\nSummary:")
print(f"  Constraints added:   {n_added}")
print(f"  Constraints skipped: {n_skipped} (noisy/inconsistent responses)")
if skipped_ids:
    print(f"  Skipped comparison IDs: {skipped_ids}")

print(f"\nFinal polytope has {polytope.n_constraints} constraints")

In [ ]:
# ============================================================================
# STEP 4: Estimate ω from polytope and recover β = V @ ω
# ============================================================================

# Get Chebyshev center (point maximizing distance to all constraint boundaries)
omega_center = polytope.chebyshev_center()

if omega_center is not None:
    print("POLYTOPE ESTIMATION RESULTS")
    print("=" * 60)
    
    # Check if oracle is in the polytope
    oracle_in_polytope = polytope.is_feasible(omega_oracle)
    print(f"Oracle ω in polytope: {oracle_in_polytope}")
    
    print(f"\nChebyshev center (ω_hat):")
    print("-" * 40)
    for name, w in zip(feature_names, omega_center):
        print(f"  {name:20s}: {w:.4f}")
    print(f"  Sum: {omega_center.sum():.4f}")
    
    # Recover signed weights: β_hat = V @ ω_hat
    beta_hat = V @ omega_center
    
    print(f"\nRecovered signed weights (β_hat = V @ ω_hat):")
    print("-" * 40)
    for name, b in zip(feature_names, beta_hat):
        print(f"  {name:20s}: {b:+.4f}")
    
    # Compare to oracle
    print(f"\n" + "=" * 60)
    print("COMPARISON TO ORACLE")
    print("=" * 60)
    print(f"\n{'Feature':<20} | {'Oracle β*':>10} | {'Estimated β':>10} | {'Error':>10}")
    print("-" * 60)
    for name, true_b, est_b in zip(feature_names, beta_oracle, beta_hat):
        err = est_b - true_b
        print(f"{name:<20} | {true_b:>+10.4f} | {est_b:>+10.4f} | {err:>+10.4f}")
    
    # Overall error metrics
    l2_error = np.linalg.norm(beta_hat - beta_oracle)
    cosine_sim = np.dot(beta_hat, beta_oracle) / (np.linalg.norm(beta_hat) * np.linalg.norm(beta_oracle))
    
    print("-" * 60)
    print(f"L2 error:        {l2_error:.4f}")
    print(f"Cosine similarity: {cosine_sim:.4f}")
    
else:
    print("ERROR: Polytope is empty or degenerate - no Chebyshev center found")